# 04 — Entrenamiento extendido de Diffusion Policy en Colab T4

**TFM:** Pose 6-DoF Transformer + Diffusion para Bin Picking

Entrena el modelo Diffusion Policy con configuración extendida frente a la versión local (M1 Pro / MPS, 30 epochs, 2 000 trayectorias):

| Parámetro | M1 Pro local | T4 Colab (este notebook) |
|-----------|-------------|--------------------------|
| Epochs | 30 | **200** |
| Trayectorias entrenamiento | 2 000 | **10 000** |
| Hardware | M1 Pro / MPS | NVIDIA T4 / CUDA |
| Tiempo aprox | ~5 min | ~30-40 min |
| MSE esperado | 0.020 | **<0.005** |

In [ ]:
# Setup Colab + clonar repo
import os, sys, subprocess
if 'google.colab' in sys.modules:
    REPO_URL = 'https://github.com/Giocrisrai/pose6dof-transformers-diffusion.git'
    REPO_DIR = '/content/repo_tfm'
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/TFM/weights'
else:
    OUTPUT_DIR = 'data/models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

import torch
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device} | torch {torch.__version__}')
if device == 'cuda':
    !nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from src.planning.diffusion_policy import SimpleDDPMScheduler, ConditionalUNet1D, DiffusionGraspPlanner

# CONFIGURACION EXTENDIDA
N_SAMPLES = 10000   # vs 2000 local
N_EPOCHS  = 200     # vs 30 local
BATCH     = 128     # vs 64 local
LR        = 2e-4
HORIZON   = 16
ACTION_DIM = 7
COND_DIM = 64
HIDDEN_DIM = 256    # vs 128 local (mas capacidad para mas datos)
TIMESTEPS = 100
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Generar dataset extendido
def generate_data(n_samples, horizon, action_dim, seed):
    np.random.seed(seed)
    planner = DiffusionGraspPlanner(action_dim=action_dim, horizon=horizon)
    trajs, conds = [], []
    for i in tqdm(range(n_samples), desc='Generando trayectorias'):
        t_obj = np.array([
            np.random.uniform(-0.4, 0.4),
            np.random.uniform(-0.4, 0.4),
            np.random.uniform(0.7, 1.0),
        ])
        cond_vec = np.zeros(64)
        cond_vec[:3] = t_obj
        cond_vec[3:6] = np.random.randn(3) * 0.1  # rotacion aleatoria pequena
        traj = planner._generate_heuristic_trajectory(t_obj)
        trajs.append(traj)
        conds.append(cond_vec)
    return np.array(trajs, dtype=np.float32), np.array(conds, dtype=np.float32)

X_train, c_train = generate_data(N_SAMPLES, HORIZON, ACTION_DIM, SEED)
X_val, c_val = generate_data(N_SAMPLES // 5, HORIZON, ACTION_DIM, SEED + 1)
print(f'Train: {X_train.shape}, Val: {X_val.shape}')

In [ ]:
class GraspDataset(Dataset):
    def __init__(self, x, c):
        self.x = torch.tensor(x); self.c = torch.tensor(c)
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.c[i]

train_loader = DataLoader(GraspDataset(X_train, c_train), batch_size=BATCH, shuffle=True, num_workers=2)
val_loader   = DataLoader(GraspDataset(X_val, c_val), batch_size=BATCH, shuffle=False)

model = ConditionalUNet1D(action_dim=ACTION_DIM, horizon=HORIZON, cond_dim=COND_DIM, hidden_dim=HIDDEN_DIM).to(device)
scheduler = SimpleDDPMScheduler(num_timesteps=TIMESTEPS)
alpha_bar = torch.tensor(scheduler.alpha_bar, dtype=torch.float32, device=device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-6)
lr_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
n_params = sum(p.numel() for p in model.parameters())
print(f'Modelo: {n_params/1e6:.2f} M params')

In [ ]:
import time
train_losses, val_losses = [], []
best_val = float('inf')
t0 = time.time()
for epoch in range(N_EPOCHS):
    model.train()
    sum_loss = 0; n = 0
    for xb, cb in train_loader:
        xb, cb = xb.to(device), cb.to(device)
        B = xb.shape[0]
        t = torch.randint(0, TIMESTEPS, (B,), device=device)
        eps = torch.randn_like(xb)
        ab_t = alpha_bar[t].view(-1,1,1)
        x_noisy = torch.sqrt(ab_t) * xb + torch.sqrt(1-ab_t) * eps
        eps_pred = model(x_noisy, t, cb)
        loss = F.mse_loss(eps_pred, eps)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        sum_loss += loss.item() * B; n += B
    train_loss = sum_loss / n
    train_losses.append(train_loss)

    # Val
    model.eval()
    with torch.no_grad():
        sum_loss = 0; n = 0
        for xb, cb in val_loader:
            xb, cb = xb.to(device), cb.to(device)
            B = xb.shape[0]
            t = torch.randint(0, TIMESTEPS, (B,), device=device)
            eps = torch.randn_like(xb)
            ab_t = alpha_bar[t].view(-1,1,1)
            x_noisy = torch.sqrt(ab_t)*xb + torch.sqrt(1-ab_t)*eps
            eps_pred = model(x_noisy, t, cb)
            sum_loss += F.mse_loss(eps_pred, eps).item() * B; n += B
        val_loss = sum_loss / n
    val_losses.append(val_loss)
    lr_sched.step()

    if val_loss < best_val:
        best_val = val_loss
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch, 'val_loss': val_loss,
                    'config': {'horizon': HORIZON, 'action_dim': ACTION_DIM,
                                'cond_dim': COND_DIM, 'hidden_dim': HIDDEN_DIM,
                                'n_samples': N_SAMPLES, 'n_epochs': N_EPOCHS}},
                   f'{OUTPUT_DIR}/diffusion_policy_extended.pth')

    if (epoch + 1) % 10 == 0:
        elapsed = (time.time() - t0) / 60
        print(f'Epoch {epoch+1:3d}/{N_EPOCHS} | Train: {train_loss:.5f} | Val: {val_loss:.5f} | Best: {best_val:.5f} | {elapsed:.1f} min')

print(f'\nFINAL: best_val_loss = {best_val:.5f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Train', color='#0098CD')
ax.plot(val_losses, label='Validation', color='#FF6B35')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
ax.set_title(f'Training extendido Diffusion Policy ({N_SAMPLES} samples, {N_EPOCHS} epochs, {HIDDEN_DIM}-dim)')
ax.legend(); ax.set_yscale('log'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/extended_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Pesos guardados en: {OUTPUT_DIR}/diffusion_policy_extended.pth')